### Feature Engineering - 

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

#### 1. Load the cleaned dataset
df = pd.read_csv('D:/Capstoneproject2-EV/notebooks/cleaned_charging_records.csv')
df['StartDatetime'] = pd.to_datetime(df['StartDatetime'])


### Before dropping the columns, we extract all necessary temporal information from StartDatetime.
#### 2. Extract basic calendar features
df['StartHour'] = df['StartDatetime'].dt.hour
df['DayOfWeek'] = df['StartDatetime'].dt.dayofweek  # 0=Monday, 6=Sunday
df['Month'] = df['StartDatetime'].dt.month
df['IsWeekend'] = df['DayOfWeek'].apply(lambda x: 1 if x >= 5 else 0)

#### 3. Cyclical Encoding for StartHour (24-hour cycle)
df['Hour_sin'] = np.sin(2 * np.pi * df['StartHour'] / 24.0)
df['Hour_cos'] = np.cos(2 * np.pi * df['StartHour'] / 24.0)

#### 4. Cyclical Encoding for DayOfWeek (7-day cycle)
df['Day_sin'] = np.sin(2 * np.pi * df['DayOfWeek'] / 7.0)
df['Day_cos'] = np.cos(2 * np.pi * df['DayOfWeek'] / 7.0)

print("Engineered temporal features successfully.")

Engineered temporal features successfully.


In [2]:
###prevent Data Leakage (Drop Future & Raw Columns)

#### Columns causing data leakage (known only after charging finishes)
leakage_cols = ['Duration', 'EndDatetime', 'EndDay', 'EndTime']

#### Raw identifier and unparsed date columns
unneeded_cols = ['UserID', 'StartDatetime', 'StartDay', 'StartTime', 'StartHour', 'DayOfWeek']

#### Drop all of them 
df_model = df.drop(columns=leakage_cols + unneeded_cols)

print("Remaining columns for modeling:")
print(df_model.columns.tolist())


Remaining columns for modeling:
['ChargerID', 'ChargerCompany', 'Location', 'ChargerType', 'Demand', 'Month', 'IsWeekend', 'Hour_sin', 'Hour_cos', 'Day_sin', 'Day_cos']


In [4]:
### Convert categorical column -(location,charge Type,charge Company) into binary columns 
## One-hot encoding
cat_cols = ['Location', 'ChargerCompany', 'ChargerType']
df_encoded = pd.get_dummies(df_model, columns=cat_cols, drop_first=True)
print("Dataset shape after encoding:", df_encoded.shape)
df_encoded.head(5)


Dataset shape after encoding: (72108, 23)


,ChargerID,Demand,Month,IsWeekend,Hour_sin,Hour_cos,Day_sin,Day_cos,Location_apartment,Location_bus garage,...,Location_hotel,Location_market,Location_public area,Location_public institution,Location_public parking lot,Location_resort,Location_restaurant,Location_sightseeing,ChargerCompany_1,ChargerType_1
0,1,20.36,9,0,-0.866025,5.000000e-01,0.433884,-0.900969,False,False,...,True,False,False,False,False,False,False,False,True,False
1,1,10.19,9,0,-0.866025,5.000000e-01,0.974928,-0.222521,False,False,...,True,False,False,False,False,False,False,False,True,False
2,1,6.78,9,0,-1.000000,-1.836970e-16,0.974928,-0.222521,False,False,...,True,False,False,False,False,False,False,False,True,False
3,1,37.65,9,0,-1.000000,-1.836970e-16,0.433884,-0.900969,False,False,...,True,False,False,False,False,False,False,False,True,False
4,1,33.81,9,1,-0.965926,2.588190e-01,-0.781831,0.623490,False,False,...,True,False,False,False,False,False,False,False,True,False


In [5]:
### Separere Feature (X) and target (y)  and Train -Test-Split
X= df_encoded.drop(columns = ['Demand'])
y = df_encoded['Demand']

### Train(80) / Test(20) split 
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")



X_train shape: (57686, 22)
X_test shape: (14422, 22)
y_train shape: (57686,)
y_test shape: (14422,)
